# From Preferences to Alignment: RLHF

> Function Calling lets a model select tools and generate arguments, but a syntactically valid call may still conflict with the user's intent. Pretraining asks which token is most likely next; alignment asks which of several plausible answers people would prefer. Likelihood and preference are different objectives.
>
> Classic RLHF converts preferences into training signals in stages. **SFT** first teaches high-quality answers. A **Reward Model** converts pairwise preferences into scores. **PPO** then updates the policy from rewards while a KL constraint limits drift from a reference model. **DPO** learns directly from chosen and rejected answers, avoiding an explicit Reward Model and online PPO. **GRPO** estimates relative advantages from a group of answers to the same question.

Many fluent answers can follow the same prompt. A pretrained model learns which token sequences are common in text, but not whether users prefer an accurate, candid answer over a smooth answer containing errors. Collecting two candidates for one prompt and marking them `chosen` and `rejected` turns preference into training data. **Alignment** studies how those signals change model behavior.

Classic RLHF trains an SFT model, Reward Model, and PPO policy in sequence. DPO optimizes the paired preference directly. We begin with a concrete pair of answers, then derive each objective.


## 1. Why Model Alignment Matters

The goal of a pretrained model is "text continuation" -- given the previous N tokens, predict the N+1th token. It has learned statistical patterns of language from massive internet text, but it doesn't understand what makes an answer "helpful."

Consider this example:

```
User: I've been feeling down lately. I don't feel like life is worth living.

Unaligned Base Model (continuation mode):
  "She stood on the rooftop, the wind blowing through her hair, thinking of
   the person who betrayed her... The crowd below gathered to watch, someone
   pulled out a phone to record..."
   ^ The model is imitating novels it saw online! Statistically this is correct --
     "feeling down" is indeed often followed by this kind of literary description.
     But this response is completely inappropriate.

Aligned Model (Helpful + Harmless):
  "I'm sorry you're going through this. Please remember: you are not alone,
   and there are many people willing to help you. You can call a crisis
   hotline. Would you like to talk about what happened?"
   ^ The model recognizes this as a cry for help and provides a helpful response.
```

This contrast reveals the fundamental problem of alignment: pretraining teaches the model "what usually follows this text," but alignment must teach it "what *should* be responded in this situation." The former is about statistical patterns; the latter is about value judgment.

The 3H principles of alignment:
- **Helpful**: Answer questions, provide useful information, stay on topic
- **Honest**: Admit when it doesn't know, don't fabricate plausible-sounding falsehoods
- **Harmless**: Refuse harmful requests, don't enable dangerous behavior

These three principles sometimes conflict. For example, if a user asks "why didn't the bomb I made go off," an honest answer would provide dangerous information -- helpful to this user but harmful to society. Alignment needs to balance the three -- usually prioritizing harmlessness.


## 2. The Full Alignment Pipeline

The entire chain is divided into several stages:

```
  Base Model (pretrained, only does continuation)
       |
       v  Stage 1: SFT
  +---------------------------+
  | Supervised learning with  |
  | high-quality conversation |
  | data. Teaches dialogue    |
  | format and basic instructions |
  +-------------+--------------+
                v
          SFT Model (can converse, but can't distinguish good from bad)
                |
          +-----+-----+
          v             v
     Stage 2a: RLHF    Stage 2b: DPO
     (classic route)    (simplified route)
          |                  |
     1. Collect preference  1. Collect preference
        data                  data
     2. Train Reward Model  2. Directly optimize
     3. PPO reinforcement      preferences
        learning              (skip RM + PPO)
          |                  |
          +------+-----------+
                 v
           Aligned Model
```

Let's work through each stage with hand calculations.


## 3. Stage 1: SFT

#### 3.1 What SFT Does

Supervised training with high-quality conversation data. The training data looks like this:

```
<|User|> What is the capital of France?
<|Assistant|> The capital of France is Paris.
<|User|> Why Paris?
<|Assistant|> Because Paris is the largest city in France and its political
center, serving as the administrative capital since the Middle Ages.
```

SFT is essentially the same as pretraining -- cross-entropy loss, predicting the next token. The only difference is that the data changes from "internet text" to "conversations."

After SFT, the model has learned:
1. Dialogue format (question-answer pairs)
2. Basic instruction following
3. Surface features of "good answers"

#### 3.2 But SFT Has a Fundamental Limitation

For questions with a single correct answer like "What is the capital of France?", SFT is sufficient.

But for open-ended questions, like "Help me write a resignation letter" -- there is no unique "standard answer." Two resignation letters can both be acceptable, but which one is better? SFT's training signal cannot distinguish between them.

This is where the next stage comes in: **having human annotators tell us which answer is better.**


#### 3.3 Loss Masking in SFT

As mentioned earlier, SFT training data is a complete conversation, typically in this format:

```text
[system prompt] [user message] [assistant response]
```


The model's task is "predict the next token." If the entire data sequence is used for training, the model will simultaneously learn to predict the system prompt, user message, and assistant response. But we only want it to learn to **generate the response part** -- after all, users don't expect the model to predict their own questions.

The solution is straightforward: mark the tokens in the prompt portion with a special label so that the loss function ignores them. In PyTorch, this special label is `-100` (the default `ignore_index` for `CrossEntropyLoss`). This is called **Loss Masking**.

The specific approach:
- Set the label for prompt tokens (system + user) to `-100`, so loss is not computed
- Set the label for response tokens (assistant) normally to the next token's ID

This way, cross-entropy loss only accumulates on response tokens, and the model only learns "how to respond" rather than being trained to reproduce the prompt.


In [ ]:
import numpy as np
# === Loss Masking demonstration ===
print("=== Loss Masking: calculate loss only on the response ===")
print()

# Simulate one SFT training sample
# Assume the tokenizer encoded the dialogue into nine Tokens
# The first five are the prompt (system plus user); the last four are the assistant response

tokens = [1, 5, 3, 8, 2, 7, 9, 4, 6]  # complete Token sequence
prompt_len = 5  # first five Tokens belong to the prompt

# Standard labels: each position label is the next Token, shifted right by one
labels_shifted = tokens[1:] + [-1]  # [5, 3, 8, 2, 7, 9, 4, 6, -1]

# Loss Masking replaces prompt labels with -100
labels_masked = labels_shifted.copy()
for i in range(prompt_len):
    labels_masked[i] = -100  # ignore the prompt region

print("tokens:      ", tokens)
print("Original labels:", labels_shifted)
print("labels(mask):", labels_masked)
print()

# Demonstrate loss calculation with simple simulated data
# Suppose the model outputs a distribution at every position; simplify this to prediction correctness
# log_probs[i] is the model log probability of tokens[i+1] at position i
np.random.seed(42)
log_probs = np.random.uniform(-3.0, -0.5, size=len(tokens))
log_probs = np.round(log_probs, 2)

print("--- Without Loss Masking ---")
total_loss = 0
count = 0
for i in range(len(tokens) - 1):  # final position has no label
    label = labels_shifted[i]
    loss_i = -log_probs[i]  # Cross-Entropy = -log(p)
    total_loss += loss_i
    count += 1
    print(f"  position {i}: Token={tokens[i]}, label={label:>3}, "
          f"log_p={log_probs[i]:.2f}, loss={loss_i:.2f}")
avg_no_mask = total_loss / count
print(f"  mean loss = {avg_no_mask:.4f}; all {count} positions participate")
print()

print("--- With Loss Masking ---")
total_loss = 0
count = 0
for i in range(len(tokens) - 1):
    label = labels_masked[i]
    if label == -100:
        print(f"  position {i}: Token={tokens[i]}, label=-100 -> skip prompt")
        continue
    loss_i = -log_probs[i]
    total_loss += loss_i
    count += 1
    print(f"  position {i}: Token={tokens[i]}, label={label:>3}, "
          f"log_p={log_probs[i]:.2f}, loss={loss_i:.2f}")
avg_masked = total_loss / count
print(f"  mean loss = {avg_masked:.4f}; only {count} response positions participate")
print()
print("Key observation: Loss Masking prevents prompt tokens from contributing gradient updates,")
print("The model learns only from the response, giving SFT a cleaner training signal.")


## 4. Stage 2: Reward Model

#### 4.1 What Preference Data Looks Like

For a given prompt, annotators see two answers and choose the better one:

```
Prompt: Help me write a resignation letter

Answer A (chosen):
  "Dear Manager: Due to personal reasons, I have decided to resign from
   my current position. Thank you for the training and opportunities the
   company has given me over the past two years. I will ensure all handover
   work is completed before my departure. Wishing the company continued success!"

Answer B (rejected):
  "Resigning? Just write 'I quit' and be done with it.
   The company won't care whether you leave anyway."
```

The annotator chose A. This is one preference data point: `(prompt, chosen, rejected)`.

#### 4.2 What the Reward Model Does

Train a model (Reward Model, RM) that takes `(prompt, answer)` as input and outputs a score r.

The training objective is simple: **make the RM give a higher score to chosen than to rejected.**

This is essentially training an "automatic scorer" to replace human annotators.

#### 4.3 Loss Function Hand Calculation -- This Is the Bradley-Terry Model


In [ ]:
import math
# === Hand-calculate Reward Model Loss ===
print("=== Reward Model loss function ===")
print()
print("Formula: L = -log(sigmoid(r_chosen - r_rejected))")
print("         sigmoid(x) = 1/(1+e^(-x))")
print()

def sigmoid(x):
    return 1 / (1 + math.exp(-x))

def reward_loss(r_chosen, r_rejected):
    diff = r_chosen - r_rejected
    prob = sigmoid(diff)
    loss = -math.log(max(prob, 1e-10))
    return loss, diff, prob

# Four scenarios
cases = [
    ("excellent: chosen >> rejected", 8.0, 2.0),
    ("acceptable: chosen slightly better", 6.0, 5.0),
    ("bad: chosen < rejected", 3.0, 7.0),
    ("disastrous: chosen << rejected", 1.0, 9.0),
]

print(f"{'Scenario':<25s} {'r_c':>6s} {'r_r':>6s} {'diff':>8s} {'sigmoid':>10s} {'loss':>10s}")
print("-" * 70)

for desc, r_c, r_r in cases:
    loss, diff, prob = reward_loss(r_c, r_r)
    print(f"{desc:<25s} {r_c:>6.1f} {r_r:>6.1f} {diff:>8.1f} {prob:>10.4f} {loss:>10.4f}")

print()
print("Interpretation:")
print("  • sigmoid(diff) is the probability that chosen should win")
print("  When r_chosen is much larger than r_rejected, sigmoid is near 1 and loss near 0: light penalty")
print("  When r_chosen is much smaller, sigmoid is near 0 and loss is large: strong penalty")
print()
print("This turns preference comparison into a binary-classification problem.")
print("  Label=1 means chosen is better; prediction is sigmoid(r_c-r_r), trained with cross-entropy")


## 5. Stage 3: PPO

With a Reward Model, we can use reinforcement learning to optimize the LLM.

#### 5.1 PPO Training Loop

```
Repeat the following steps:
  1. Take a batch of prompts (e.g., 256)
  2. LLM generates a response for each prompt
  3. RM scores each (prompt, answer) pair -> get reward
  4. Use PPO to update the LLM -- increase high-scoring tokens, decrease low-scoring ones
  5. But don't stray too far -- add KL penalty (prevent the model from forgetting what SFT taught)
```

#### 5.2 PPO's Core Mechanism: Clipping

The most critical design in PPO is a mechanism called **clip**. Here's the idea in one page:

```
ratio = new model's probability of choosing this token / old model's probability of choosing this token

If ratio = 1.5 -> old model chose this token with probability 2%, new model 3%
If ratio = 0.8 -> old model chose this token with probability 5%, new model 4%

Clip's role: constrain ratio to [1-epsilon, 1+epsilon] (typically epsilon=0.2)
  -> Prevent overly aggressive single-step updates
  -> Like a steering wheel limiter -- prevents you from turning too hard and rolling over
```


In [ ]:
import numpy as np

# === Hand-calculate the PPO clipping mechanism ===
print("=== PPO Clipping ===")
print()

def ppo_loss(ratio, advantage, epsilon=0.2):
    """
    Calculate PPO surrogate loss.
    ratio = pi_new / pi_old, the new-to-old policy probability ratio
    advantage = RM_score - baseline, how much better this response is than average
    """
    # Loss without clipping
    unclipped = ratio * advantage
    # Clipped loss, restricting ratio to [0.8, 1.2]
    clipped = np.clip(ratio, 1-epsilon, 1+epsilon) * advantage
    # PPO takes the conservative option: min for advantage>0, max for advantage<0
    # Equivalently, use -min(unclipped, clipped) as the surrogate
    return -min(unclipped, clipped)


ratios = [0.5, 0.7, 0.9, 1.0, 1.1, 1.3, 1.5, 2.0]

print("When advantage > 0, this is a good response whose probability should increase:")
print(f"{'ratio':>8s} {'unclipped':>12s} {'clipped':>12s} {'loss':>10s} {'Note'}")
print("-" * 60)
for r in ratios:
    unclipped = r * 1.0
    clipped = min(r, 1.2) * 1.0
    loss = -min(unclipped, clipped)
    note = ""
    if r > 1.2:
        note = "<- clipped; do not let it grow further"
    print(f"{r:>8.2f} {unclipped:>12.2f} {clipped:>12.2f} {loss:>10.2f} {note}")

print()
print("When advantage < 0, this is a bad response whose probability should decrease:")
print(f"{'ratio':>8s} {'unclipped':>12s} {'clipped':>12s} {'loss':>10s} {'Note'}")
print("-" * 60)
for r in ratios:
    unclipped = r * (-1.0)
    clipped = max(r, 0.8) * (-1.0)
    loss = -min(unclipped, clipped)
    note = ""
    if r < 0.8:
        note = "<- clipped; do not let it shrink further"
    print(f"{r:>8.2f} {unclipped:>12.2f} {clipped:>12.2f} {loss:>10.2f} {note}")

print()
print("Interpretation:")
print("  • advantage>0, good Token: PPO encourages a larger ratio, but no more than 1.2")
print("  • advantage<0, bad Token: PPO encourages a smaller ratio, but no less than 0.8")
print("  • clipping is a safety net that limits one-step updates and prevents training collapse")


#### 5.3 The Complete PPO Loss Formula

```
L_PPO = -E[ min(ratio x A, clip(ratio, 1-epsilon, 1+epsilon) x A) ]
        + beta x KL(pi_theta || pi_SFT)

Where:
  ratio = pi_theta(token|prompt) / pi_old(token|prompt)
  A = advantage = RM_score - baseline
  KL term = distributional difference between the new model and the SFT model
  beta = weight of the KL penalty (hyperparameter, typically 0.01-0.1)
```

The KL penalty is an easily overlooked but critically important part of PPO. Its role is not to improve performance, but to prevent the model from degrading.

The problem lies with the Reward Model. The RM is a scorer trained on human-annotated data -- it is only an approximate model of human preferences, not a perfect judge. If we optimize the LLM using only RM scores, the model will find loopholes in the RM's scoring mechanism and generate "score-gaming" responses that have high RM scores but terrible quality. This is called **Reward Hacking**.

For example, the RM might learn to give high scores to long responses (because annotators tend to choose more detailed answers). If optimized solely by RM scores, the model would learn to make responses endlessly long, even repeating content to pad the length -- the RM score would be high, but the response quality would be abysmal.

The KL penalty prevents this: it limits how far the new model can deviate from the SFT model. The SFT model may not distinguish good from bad, but at least it doesn't produce nonsense. The KL penalty essentially says: "You can optimize RM scores, but you can't deviate too far from the SFT model." The formula here is a teaching version; real PPO/RLHF typically also includes value functions, GAE, token-level advantages, clip/value losses, and other details. The larger beta is, the stronger the constraint, and the more conservative the model becomes.


## 6. DPO

#### 6.1 Problems with RLHF

RLHF is powerful, but engineering-wise it is very complex:
- Requires training a separate Reward Model (typically as many parameters as the main model)
- During PPO training, 4 models are live simultaneously: Actor (being trained), Reference (frozen SFT), Reward Model, Critic (value network)
- Many hyperparameters: KL coefficient, clip range, learning rate, rollout count...
- Unstable: PPO is notoriously "hard to tune"

In 2023, DPO (Direct Preference Optimization) proposed a simpler approach.

#### 6.2 The Mathematical Insight Behind DPO

The optimal policy of RLHF can be written as a function of the Reward Model and the reference policy. Through algebraic transformation, the Reward Model can be "eliminated" -- the result is the DPO loss:

```
L_DPO = -log( sigma(
    beta x log( pi_theta(chosen) / pi_ref(chosen) )
  - beta x log( pi_theta(rejected) / pi_ref(rejected) )
))
```

This formula looks intimidating, but the intuition is simple:

```
pi_theta(chosen) up   -> loss down  (good response, increase its probability!)
pi_theta(rejected) up -> loss up    (bad response, decrease its probability!)
pi_ref as denominator  -> prevents drifting too far (similar role to KL penalty)
beta -> controls "how aggressively" to change (larger beta = more aggressive)
```


In [ ]:
import math

# === Hand-calculate DPO Loss ===
print("=== DPO Loss intuition ===")
print()

def dpo_loss(logp_chosen, logp_rejected, logp_ref_chosen, logp_ref_rejected, beta=0.5):
    """
    Simplified DPO loss.
    logp is the log probability of generating a response.
    A larger, less negative value means the model favors that response more.
    """
    # Improvement of chosen relative to the reference
    chosen_improvement = logp_chosen - logp_ref_chosen
    # Improvement of rejected relative to the reference; we want this to worsen
    rejected_improvement = logp_rejected - logp_ref_rejected
    
    # Core: chosen should improve more than rejected
    diff = beta * (chosen_improvement - rejected_improvement)
    
    # Apply sigmoid, then -log
    prob = 1 / (1 + math.exp(-diff))
    loss = -math.log(max(prob, 1e-10))
    
    return loss, chosen_improvement, rejected_improvement, diff, prob


scenarios = [
    ("ideal: chosen up, rejected down",    -1.0, -5.0, -3.0, -3.0),
    ("okay: both rise, chosen rises more", -1.0, -2.5, -3.0, -3.0),
    ("bad: chosen down, rejected up",      -5.0, -1.0, -3.0, -3.0),
    ("both rise, chosen rises more",       -0.5, -2.0, -3.0, -3.0),
    ("both fall, chosen falls less",       -4.5, -6.0, -3.0, -3.0),
]

print(f"{'Scenario':<28s} {'chosen_imp':>10s} {'rej_imp':>10s} {'diff':>10s} {'loss':>10s}")
print("-" * 74)

for desc, lc, lr, ref_c, ref_r in scenarios:
    loss, ci, ri, diff, prob = dpo_loss(lc, lr, ref_c, ref_r)
    print(f"{desc:<28s} {ci:>+10.2f} {ri:>+10.2f} {diff:>10.2f} {loss:>10.4f}")

print()
print("Core logic: DPO ignores absolute values and compares chosen's improvement against rejected's")
print("  More chosen improvement and less rejected improvement -> larger diff -> lower loss")
print("  Even if both chosen and rejected probabilities fall because the model is forgetting,")
print("  loss still falls whenever chosen falls less than rejected, giving diff > 0")


## 7. RLHF and DPO

| Dimension | RLHF (PPO) | DPO |
|:---|:---|:---|
| **Number of models** | 4 (Actor, Ref, RM, Critic) | 2 (Train, Ref) |
| **Training mode** | Online (each step requires model to generate new responses) | Offline (uses pre-collected preference pairs) |
| **Stability** | Poor (PPO is notoriously hard to tune) | Good (just supervised learning) |
| **Reward hacking** | High risk (RM is a fixed target) | Low risk (no RM to exploit) |
| **Theoretical ceiling** | Potentially higher (more online exploration) | Limited by preference data |
| **Debugging difficulty** | Hard (multiple components, hard to locate which one is broken) | Easy (one loss, debug like SFT) |
| **Who uses it** | OpenAI (GPT-4), Anthropic (Claude) | Open-source community (Zephyr, Qwen, etc.) |

**Recommendations**:
- Small teams, fast iteration -> DPO (less effort, comparable results)
- Large teams, pursuing maximum quality -> RLHF (large investment, higher ceiling)
- Middle ground -> Iterative DPO (multiple rounds of DPO, generating new preference pairs with the current model each round)


## 8. LLaMA 2's Complete Alignment Pipeline

```
Stage 1 -- Pretraining:
  LLaMA 2 Base, 2T tokens
  -> Can continue text, cannot converse

Stage 2 -- SFT:
  ~27K high-quality human-annotated conversations
  Trained for 2 epochs
  -> Can converse, follows instructions

Stage 3 -- Collect Preference Data:
  ~1M+ (prompt, chosen, rejected) pairs
  Annotators compare responses pairwise
  -> Human preference dataset

Stage 4a -- Reward Model:
  Initialized from SFT model
  Trained to score using preference data
  -> Automatic scorer

Stage 4b -- PPO:
  Score with RM -> PPO optimizes SFT model
  5 rounds of iteration
  -> LLaMA 2 Chat
```


## 9. Limitations and Applicable Scenarios of Alignment

RLHF/DPO makes models safer and more helpful, but also introduces new problems:

1. **Over-refusal**: The model may refuse harmless requests. For example, asking "how to perform CPR" might be misjudged as harmful medical advice and refused -- the model has over-associated "medical" with "harmful."
2. **Sycophancy**: If a user makes an error (e.g., "1+1=3"), the model doesn't correct them but instead agrees. This happens because annotators typically prefer responses that "don't contradict the user," and the RM has learned this preference.
3. **Style homogenization**: All responses become "polite safety-speak" -- starting with "Of course" or "I'd be happy to help" and ending with "I hope this helps." The diverse language style of the pretrained model is lost.
4. **Knowledge loss**: During alignment, the model may "forget" knowledge learned during pretraining. Because alignment data is typically concentrated on conversational and Q&A scenarios, the model's grasp of factual knowledge may degrade.

These problems are still actively researched, and there are no perfect solutions yet.

It's also worth emphasizing that alignment is not universally applicable. For tasks like code completion, translation, and text summarization, the definition of "good" is determined by objective downstream metrics (compilation pass rate, BLEU score, ROUGE score), and human preferences play a limited role. RLHF/DPO is primarily applicable to open-domain dialogue and instruction-following scenarios -- in these contexts, "good" is indeed subjective and needs human definition.


## 10. Training an Aligned Model with GRPO

The preceding sections calculated SFT, Reward Model, PPO, and DPO losses by hand. A real training run must still move a randomly initialized policy toward high-reward behavior one update at a time.

We use a clean task—reversing a string—to run the complete GRPO loop. A program can score answers exactly, so no Reward Model is needed and we can focus on reinforcement learning. GRPO shares PPO's clipping mechanism; its main difference is how it obtains the baseline.


### Task: Reverse a String

Given a three-character string drawn from `abcdef`, the model must output its reverse. Input `cab` has answer `bac`. The task is deterministic, so a program supplies the reward without human labels or a trained Reward Model.

Real RLHF rewards are more complex. Researchers often begin with programmable rewards to validate the learning loop before adding a learned scorer.

The sequence format is `<BOS> input <SEP> output <EOS>`. The prompt ends at `<SEP>`, and the model must generate the reversed suffix.


In [ ]:
import numpy as np
import torch

torch.manual_seed(42)
np.random.seed(42)
# Small models are slower with many threads because scheduling overhead exceeds compute, so limit threads
torch.set_num_threads(4)

CHARS = "abcdef"
SPECIAL = ["<BOS>", "<SEP>", "<EOS>"]
itos = SPECIAL + list(CHARS)            # vocabulary: three boundary symbols plus six characters
stoi = {ch: i for i, ch in enumerate(itos)}
VOCAB = len(itos)                        # 9
BOS, SEP, EOS = stoi["<BOS>"], stoi["<SEP>"], stoi["<EOS>"]


def make_pair(rng):
    """Generate a three-character string and its reversal; return (input IDs, reversed IDs)."""
    chars = [rng.choice(list(CHARS)) for _ in range(3)]
    inp = [stoi[c] for c in chars]
    return inp, inp[::-1]                # [::-1] reverses a list


def build_prompt(rng):
    '"""prompt = <BOS> Input <SEP>"""'
    inp, _ = make_pair(rng)
    return [BOS] + inp + [SEP]


def build_full(rng):
    """Complete sequence = <BOS> input <SEP> reversal <EOS>."""
    inp, out = make_pair(rng)
    return [BOS] + inp + [SEP] + out + [EOS]


rng = np.random.RandomState(0)
print("A few training samples, prompt | correct reversal:")
for _ in range(3):
    full = build_full(rng)
    print("  ", " ".join(itos[i] for i in full))


### A Minimal Decoder-Only Model

The model reuses the Part 1 structure: token embedding, position embedding, two Transformer Blocks, and a vocabulary projection. Capacity is not the goal; it only needs to learn patterns in nine-token sequences. With about 100K parameters, one round trains on CPU in seconds.


In [ ]:
import math
import torch.nn as nn
import torch.nn.functional as F


class TinyLM(nn.Module):
    'Vocabulary'
    def __init__(self, d=64, n_head=2, n_layer=2, max_len=12):
        super().__init__()
        self.d, self.n_head, self.n_layer = d, n_head, n_layer
        self.tok = nn.Embedding(VOCAB, d)
        self.pos = nn.Embedding(max_len, d)
        # Per-layer parameters: QKV projections, output projection, two FFN layers, and two LayerNorms
        self.qkv = nn.ModuleList([nn.Linear(d, 3 * d) for _ in range(n_layer)])
        self.o = nn.ModuleList([nn.Linear(d, d) for _ in range(n_layer)])
        self.ff1 = nn.ModuleList([nn.Linear(d, 4 * d) for _ in range(n_layer)])
        self.ff2 = nn.ModuleList([nn.Linear(4 * d, d) for _ in range(n_layer)])
        self.ln1 = nn.ModuleList([nn.LayerNorm(d) for _ in range(n_layer)])
        self.ln2 = nn.ModuleList([nn.LayerNorm(d) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, VOCAB, bias=False)

    def forward(self, idx):
        B, T = idx.shape                       # idx: integer Token IDs [batch, seq]
        dh = self.d // self.n_head
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        mask = torch.tril(torch.ones(T, T, device=idx.device))  # lower triangle is visible
        for li in range(self.n_layer):
            h = self.ln1[li](x)                # Pre-LN plus causal multi-head Self-Attention
            q, k, v = self.qkv[li](h).split(self.d, dim=-1)
            q = q.view(B, T, self.n_head, dh).transpose(1, 2)
            k = k.view(B, T, self.n_head, dh).transpose(1, 2)
            v = v.view(B, T, self.n_head, dh).transpose(1, 2)
            scores = (q @ k.transpose(-2, -1)) / math.sqrt(dh)
            scores = scores.masked_fill(mask == 0, float("-inf"))
            y = F.softmax(scores, dim=-1) @ v
            y = y.transpose(1, 2).contiguous().view(B, T, self.d)
            x = x + self.o[li](y)
            x = x + self.ff2[li](F.relu(self.ff1[li](self.ln2[li](x))))  # Pre-LN + FFN
        return self.head(self.ln_f(x))         # [batch, seq, vocab]


model = TinyLM()
print(f"Parameter count: {sum(p.numel() for p in model.parameters()):,}")


### Three Utility Functions

SFT evaluation and GRPO training both need three operations: calculate a generated sequence's log probability, sample from a prompt, and compare the result with the correct answer to obtain reward.

`seq_logprobs` must align positions carefully. Logits at position $t$ predict token $t+1$, so use `logits[:, :-1]` with `targets[:, 1:]`, then retain only the generated portion.


In [ ]:
def seq_logprobs(model, full_ids, gen_start):
    """Calculate model log probability for every generated Token, returning [batch, n_gen].
    gen_start is the prompt length; Tokens after it belong to generation.
    """
    logp = F.log_softmax(model(full_ids), dim=-1)          # [B, T, V]
    shifted = logp[:, :-1, :].gather(
        2, full_ids[:, 1:].unsqueeze(-1)).squeeze(-1)      # [B, T-1]
    return shifted[:, gen_start - 1:]                       # [B, n_gen]


@torch.no_grad()
def rollout(model, prompt_ids, n_gen, temperature=1.0):
    """Autoregressively sample n_gen Tokens from a prompt; return [B, P+n_gen]."""
    out = prompt_ids.clone()
    for _ in range(n_gen):
        logits = model(out)[:, -1, :] / temperature
        nxt = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
        out = torch.cat([out, nxt], dim=1)
    return out


def reward_of(full_ids, prompt_len, n_gen):
    """Compare generated characters with the correct reversal and return one score from 0 to 1 per sample.
    Truncate early at <EOS>; score is the fraction of positions matching.
    """
    scores = []
    for b in range(full_ids.shape[0]):
        prompt = full_ids[b, :prompt_len].tolist()
        sep_pos = prompt.index(SEP)
        target = prompt[1:sep_pos][::-1]            # input reversal is the correct answer
        gen = full_ids[b, prompt_len:prompt_len + n_gen].tolist()
        if EOS in gen:
            gen = gen[:gen.index(EOS)]
        correct = sum(1 for i, t in enumerate(target)
                      if i < len(gen) and gen[i] == t)
        scores.append(correct / len(target))
    return torch.tensor(scores)


# Sample once from the untrained model to inspect reward
prompts = torch.tensor([build_prompt(np.random.RandomState(i)) for i in range(4)])
full = rollout(model, prompts, n_gen=4)
print("One sample from the untrained model, nearly all wrong:")
for b in range(4):
    print("  ", " ".join(itos[i] for i in full[b].tolist()))
print("Reward:", reward_of(full, prompt_len=5, n_gen=4).tolist())


### SFT Initialization

GRPO rarely learns from a completely random policy: almost every reversal is wrong, all responses in a group receive the same reward, and every advantage becomes zero. We therefore use a small amount of SFT so the model can sometimes solve the task, then let GRPO push it toward near-perfect accuracy.

The SFT model also becomes the reference policy $\pi_{ref}$. GRPO's KL penalty compares the current policy with this reference and prevents excessive drift. To make GRPO's effect visible, SFT intentionally stops near reward 0.5 rather than 1.0.


In [ ]:
@torch.no_grad()
def eval_reward(model, n=64):
    """Sample n prompts and return the average fraction of correctly generated reversal characters, from 0 to 1."""
    prompts = torch.tensor([build_prompt(np.random.RandomState(i)) for i in range(n)])
    full = rollout(model, prompts, n_gen=4)
    return reward_of(full, prompt_len=5, n_gen=4).mean().item()


rng = np.random.RandomState(0)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

print("=== SFT, a weak starting point trained for only 40 steps ===")
for step in range(40):
    batch = torch.tensor([build_full(rng) for _ in range(32)])
    logits = model(batch)
    loss = F.cross_entropy(logits[:, :-1].reshape(-1, VOCAB),
                           batch[:, 1:].reshape(-1))
    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    if step % 10 == 0:
        print(f"  step {step:3d}  loss {loss.item():.3f}  reward {eval_reward(model):.3f}")

print(f"\nReward after SFT = {eval_reward(model):.3f}")
print("Key observation: the model moves from nearly random reward around 0.1 to roughly capable around 0.5, but still errs often.")


### The GRPO Training Loop

Each GRPO step performs five operations:

1. **Sample** a group of answers for each prompt from the current policy.
2. **Score** every answer with the programmable reward.
3. Compute **group-relative advantage** by subtracting the group's mean and dividing by its standard deviation.
4. Apply the **clipped update** to raise high-reward answer probability and lower low-reward probability.
5. Add a **KL penalty** that limits drift from $\pi_{ref}$.

Step 3 distinguishes GRPO from classic PPO. PPO trains a separate value network for its baseline; GRPO compares the $G$ answers to the same problem and needs no critic. The clipped policy update remains the same.


In [ ]:
# Reference policy: freeze the post-SFT model and use it only for KL
ref_model = TinyLM()
ref_model.load_state_dict(model.state_dict())
for p in ref_model.parameters():
    p.requires_grad_(False)

opt = torch.optim.AdamW(model.parameters(), lr=1e-3)


def grpo_step(group=4, n_gen=4, beta=0.01, eps=0.2):
    """Run one GRPO update and return (mean reward, KL)."""
    base = [build_prompt(np.random) for _ in range(8)]
    prompts = torch.tensor(base * group)             # repeat each prompt group times
    B, P = prompts.shape

    with torch.no_grad():
        full = rollout(model, prompts, n_gen=n_gen)         # 1. sample
        old_logp = seq_logprobs(model, full, P)             # old-policy logp
        ref_logp = seq_logprobs(ref_model, full, P)         # reference-policy logp
        rewards = reward_of(full, P, n_gen).view(8, group)  # 2. score

    # 3. Group advantage: subtract each prompt group mean and divide by standard deviation
    adv = (rewards - rewards.mean(1, keepdim=True)) / (
        rewards.std(1, keepdim=True) + 1e-4)
    adv = adv.view(B, 1)

    new_logp = seq_logprobs(model, full, P)         # current-policy logp with gradient
    ratio = (new_logp - old_logp).exp()             # π_new / π_old
    # 4. Clipped surrogate loss, as hand-calculated in Section 5
    surr1 = ratio * adv
    surr2 = torch.clamp(ratio, 1 - eps, 1 + eps) * adv
    policy_loss = -torch.min(surr1, surr2).mean()
    # 5. KL penalty using Schulman k3: exp(d)-d-1, where d = logp_ref - logp_new
    d = ref_logp - new_logp
    kl = (d.exp() - d - 1).mean()
    loss = policy_loss + beta * kl

    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    return rewards.mean().item(), kl.item()


In [ ]:
import copy
import time

sft_snapshot = copy.deepcopy(model)   # snapshot before GRPO for final comparison

print('"=== GRPO training ==="')
history_r, history_kl = [], []
t0 = time.time()
for step in range(100):
    mr, kl = grpo_step()
    history_r.append(mr)
    history_kl.append(kl)
    if step % 10 == 0 or step == 99:
        print(f"  step {step:3d}  reward {mr:.3f}  KL {kl:.3f}")

print(f"\nReward: {history_r[0]:.3f} -> {history_r[-1]:.3f}  ({time.time()-t0:.1f}s)")


### Reward Curves

The left plot shows mean reward; the right shows KL divergence from the reference policy. KL rises as reward improves. The KL penalty keeps the policy moving toward better answers without allowing it to travel arbitrarily far from its starting behavior.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(history_r, color="steelblue")
axes[0].set_xlabel("GRPO step")
axes[0].set_ylabel("mean reward")
axes[0].set_title("Reward climbs during GRPO")
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_kl, color="darkorange")
axes[1].set_xlabel("GRPO step")
axes[1].set_ylabel("KL(policy || ref)")
axes[1].set_title("Policy drifts away from reference")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Key observation: reward rises from about 0.5 toward 1.0 while KL rises too; the policy is leaving the reference model.")


In [ ]:
@torch.no_grad()
def show_reversals(model, seeds):
    for s in seeds:
        torch.manual_seed(s)
        pr = torch.tensor([build_prompt(np.random.RandomState(s))])
        full = rollout(model, pr, n_gen=4)[0].tolist()
        sep_pos = full.index(SEP)
        inp = "".join(itos[i] for i in full[1:sep_pos])
        target = inp[::-1]
        gen = full[5:9]
        if EOS in gen:
            gen = gen[:gen.index(EOS)]
        gen = gen[:len(target)]              # inspect only target-length positions, ignoring extra suffix
        out = "".join(itos[i] for i in gen)
        mark = "OK " if out == target else "X  "
        print(f"  {mark}input {inp}  ->  generated {out}  (correct {target})")


print("=== Before GRPO, after SFT only ===")
show_reversals(sft_snapshot, seeds=[0, 1, 2, 3, 4])
print("=== GQA implementation ===")
show_reversals(model, seeds=[0, 1, 2, 3, 4])


### What the Toy Task Simplifies

Reward rises from about 0.5 to nearly 1.0 within 100 steps. Compared with real RLHF, this experiment simplifies several pieces:

- Reward is calculated by a program; real RLHF usually trains a Reward Model from preference data first.
- The vocabulary contains nine tokens and sequences contain nine positions; real sequences are much longer and rewards are less exact.
- The essential GRPO mechanisms—group advantage, clipped surrogate loss, and KL penalty—remain the same.

The PPO, clipping, and KL equations calculated earlier have now become a runnable loop with a visible reward increase.


## Summary

Confirm that you understand these points (check in order):

1. ✅ Why alignment is needed: pretraining only teaches "how to speak," alignment teaches "how to speak appropriately"
2. ✅ 3H principles: Helpful (useful) + Honest (truthful) + Harmless (safe)
3. ✅ SFT: Uses high-quality conversations to teach dialogue format, but cannot distinguish good from bad
4. ✅ Preference data: (prompt, chosen, rejected) pairs, selected by annotators
5. ✅ Reward Model Loss: -log(σ(r_chosen - r_rejected)), turning preferences into classification
6. ✅ PPO Clip: ratio = π_new/π_old, constrained to [0.8, 1.2] to prevent overly aggressive updates
7. ✅ KL penalty: Prevents the model from gaming the score and drifting (reward hacking)
8. ✅ DPO: Skips RM+PPO, directly optimizes preference pairs -> simpler, comparable results
9. ✅ RLHF vs DPO: RLHF has a higher ceiling but is engineering-heavy; DPO offers better cost-effectiveness

**One-line summary**: Alignment = SFT foundation (learn to converse) -> RM establishes evaluation criteria -> PPO/DPO optimizes against those criteria. DPO makes this process accessible to small teams, not just large companies.


## Exercises

> You can ask an AI to explain the ideas or check your direction, but don't have it "solve the exercise" for you.

**Exercise 1: Hand-calculate Reward Model Loss**

The Reward Model uses the Bradley-Terry model, with the loss formula: $$L = -\log(\sigma(r_{chosen} - r_{rejected}))$$ Given $r_{chosen} = 2.0$ and $r_{rejected} = -1.0$. Compute the loss by hand.

Hint: $\sigma(x) = 1 / (1 + e^{-x})$. First compute $r_{chosen} - r_{rejected} = 3.0$, then $\sigma(3.0)$, and finally $-\log(\sigma(3.0))$.


In [ ]:
# Exercise 1: hand-calculate Reward Model Loss
import math
r_chosen = 2.0
r_rejected = -1.0
# TODO: calculate reward difference between chosen and rejected
diff = None  # r_chosen - r_rejected
# TODO: calculate sigmoid(diff)
sigmoid_val = None  # 1 / (1 + exp(-diff))
# TODO: calculate loss = -log(sigmoid_val)
loss = None  # -log(sigmoid_val)
assert diff is not None, 'Please replace the placeholder before running the assertion.'
assert sigmoid_val is not None, 'Please replace the placeholder before running the assertion.'
assert loss is not None, 'Please replace the placeholder before running the assertion.'
expected_diff = r_chosen - r_rejected
expected_sig = 1 / (1 + math.exp(-expected_diff))
expected_loss = -math.log(expected_sig)
assert diff == expected_diff, f"diff should be {expected_diff}"
assert abs(sigmoid_val - expected_sig) < 0.001, f"sigmoid should be {expected_sig:.4f}"
assert abs(loss - expected_loss) < 0.01, f"loss should be {expected_loss:.4f}"
print(f"✅ Exercise 1 passed")
print(f"   diff = {diff:.1f}")
print(f"   sigmoid({diff:.1f}) = {sigmoid_val:.4f}")
print(f"   loss = -log({sigmoid_val:.4f}) = {loss:.4f}")
print("   When chosen greatly exceeds rejected, the difference is large and loss approaches zero.")


**Exercise 2: PPO Clip Mechanism**

The core of PPO is the clip function, which limits the magnitude of policy updates: $\text{clip}(\text{ratio}, 1-\epsilon, 1+\epsilon)$. Given $\epsilon = 0.2$, compute the clipped result for the following three ratio values:

1. $\text{ratio} = 0.5$
2. $\text{ratio} = 1.1$
3. $\text{ratio} = 2.0$

Hint: clip simply constrains the value to the range $[1-\epsilon, 1+\epsilon] = [0.8, 1.2]$.


In [ ]:
# Exercise 2: PPO clipping
epsilon = 0.2
ratio_1 = 0.5
ratio_2 = 1.1
ratio_3 = 2.0
# TODO: clip the three ratio values to [0.8, 1.2]
clipped_1 = None  # calculate here
clipped_2 = None  # calculate here
clipped_3 = None  # calculate here
assert clipped_1 is not None, 'Please replace the placeholder before running the assertion.'
assert clipped_2 is not None, 'Please replace the placeholder before running the assertion.'
assert clipped_3 is not None, 'Please replace the placeholder before running the assertion.'
assert clipped_1 == 0.8, f"ratio=0.5 should clip to 0.8; you got {clipped_1}"
assert clipped_2 == 1.1, f"ratio=1.1 lies inside [0.8,1.2] and should not change"
assert clipped_3 == 1.2, f"ratio=2.0 should clip to 1.2; you got {clipped_3}"
print(f"\✅ Exercise 2 passed")
print(f"   ratio=0.5 -> clip=0.8, pulling an overly small update back to the lower bound")
print(f"   ratio=1.1 -> clip=1.1, unchanged inside the safe range")
print(f"   ratio=2.0 -> clip=1.2, pulling an overly large update back to the upper bound")
print("   PPO clipping prevents an excessive one-step update and keeps training stable.")


**Exercise 3: DPO Loss Analysis**

The DPO loss formula is: $$L = -\log\frac{1}{1 + e^{-\beta(\Delta_{chosen} - \Delta_{rejected})}}$$ Given $\beta = 0.5$, $\Delta_{chosen} = 0.8$, and $\Delta_{rejected} = 0.2$. Compute the DPO loss.

Hint: First compute $\beta(\Delta_{chosen} - \Delta_{rejected}) = 0.5 \times 0.6 = 0.3$, then apply the sigmoid.


In [ ]:
# Exercise 3: analyze DPO Loss
import math
beta = 0.5
delta_chosen = 0.8
delta_rejected = 0.2
# TODO: calculate beta * (delta_chosen - delta_rejected)
margin = None  # calculate here
# TODO: calculate loss = -log(sigmoid(margin))
sigmoid_m = None  # sigmoid(margin)
loss = None       # -log(sigmoid_m)
assert margin is not None, 'Please replace the placeholder before running the assertion.'
assert loss is not None, 'Please replace the placeholder before running the assertion.'
expected_margin = beta * (delta_chosen - delta_rejected)
expected_sig = 1 / (1 + math.exp(-expected_margin))
expected_loss = -math.log(expected_sig)
assert abs(margin - expected_margin) < 0.001, f"margin should be {expected_margin}"
assert abs(loss - expected_loss) < 0.01, f"loss should be {expected_loss:.4f}"
print(f"✅ Exercise 3 passed")
print(f"   margin = β × (Δ_chosen - Δ_rejected) = {margin:.2f}")
print(f"   loss = -log(σ({margin:.2f})) = {loss:.4f}")
print("   DPO measures relative improvement: the more chosen improves over rejected, the smaller the loss.")


**Exercise 4: Calculate Group Advantage by Hand**

GRPO subtracts the group mean and divides by the group standard deviation. For rewards $[0.0, 1.0, 0.0, 1.0]$, calculate the advantage of the first answer.

Hint: the mean is 0.5; compute the standard deviation, then use $(0.0-\mathrm{mean})/\mathrm{std}$.


In [ ]:
# Exercise 4: hand-calculate group advantage
import numpy as np
rewards = np.array([0.0, 1.0, 0.0, 1.0])
mean = rewards.mean()
std = rewards.std()
# TODO: calculate advantage for the first response, reward=0.0
adv_0 = None  # (rewards[0] - mean) / std

assert adv_0 is not None, 'Please replace the placeholder before running the assertion.'
expected = (rewards[0] - mean) / std
assert abs(adv_0 - expected) < 1e-6, f"advantage should be {expected:.4f}"
print(f"✅ Exercise 4 passed: mean={mean}, std={std:.4f}")
print(f"   advantage(reward=0.0) = {adv_0:.4f}; negative means this response is below the group average")


## References

- Ouyang et al., [Training language models to follow instructions with human feedback (InstructGPT)](https://arxiv.org/abs/2203.02155), 2022 — the complete SFT + RM + PPO pipeline
- Rafailov et al., [Direct Preference Optimization](https://arxiv.org/abs/2305.18290), 2023 — optimizes preferences directly without an explicit RM
- Shao et al., [DeepSeekMath / GRPO](https://arxiv.org/abs/2402.03300), 2024 — group-relative policy optimization used in Section 10
